In [ ]:
# S6E3 Notebook for Churn Prediction

import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from tqdm import tqdm



In [ ]:
# 1. 读取数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')


In [ ]:
# 2. 特征工程：Bi/Tri-gram Target Encoding
# 自定义特征工程函数
def bi_tri_target_encoding(df_train, df_test, cols, n=2):
    # 这里可以添加 Bi-gram 或 Tri-gram 的编码逻辑
    encoder = ce.TargetEncoder(cols=cols)
    df_train[cols] = encoder.fit_transform(df_train[cols], df_train['target'])
    df_test[cols] = encoder.transform(df_test[cols])
    return df_train, df_test


In [ ]:
# 3. 训练模型：XGBoost
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
}

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(train, train['target'])):
    X_train, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_train, y_val = X_train['target'], X_val['target']
    
    # 特征选择
    features = [col for col in train.columns if col not in ['target', 'id']]

    # Bi/Tri-gram Encoding
    X_train, X_val = bi_tri_target_encoding(X_train, X_val, features)

    # 训练 XGBoost
    dtrain = xgb.DMatrix(X_train[features], label=y_train)
    dval = xgb.DMatrix(X_val[features], label=y_val)

    model = xgb.train(params, dtrain, num_boost_round=1000, early_stopping_rounds=50, evals=[(dval, 'eval')])

    # 获取预测结果
    oof_preds[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))
    test_preds += model.predict(xgb.DMatrix(test[features]), iteration_range=(0, model.best_iteration)) / 5



In [ ]:
# 4. 保存预测结果
sub = pd.DataFrame({
    'id': test['id'],
    'target': test_preds
})
sub.to_csv('../submission.csv', index=False)